# Lab 2: Simulating Decentralized Agreement and Control
**Course:** Blockchain and its Application  
**Instructor:** Vishnu Suresh  
**Student Notebook:** `akcasu_lab2.ipynb`  
**Environment:** Python 3 (Jupyter Notebook)

---
### Lab Overview
Decentralization distributes particular decisions across defined actors. This exercise focuses on the decision to accept an update. Five universities operate validators and jointly maintain a credential ledger. The code simulates quorum approval, crash faults, Byzantine behavior, network partitions, Sybil attacks, and unequal voting power to connect experimental results with core decentralization principles.

## Part 1: Trust model and quorum approval
Five known universities operate validator nodes. A university submits a credential proposal, every validator applies the same local rule, and at least four validators must approve before the consortium accepts the proposal.

In [1]:
# Step 1: Define the participants and proposal
validators = [
    "University A", "University B", "University C",
    "University D", "University E",
]
proposal = {
    "student": "Marta Kowalska",
    "credential": "Blockchain Fundamentals",
    "issuer": "University A",
}

# Step 2: Define the local validation rule
def validate_proposal(proposal):
    required = {"student", "credential", "issuer"}
    missing = required - set(proposal)
    if missing:
        return False, f"Missing fields: {sorted(missing)}"
    if proposal["issuer"] not in validators:
        return False, "Issuer is not a consortium member"
    return True, "Proposal follows the rules"

# Step 3: Collect votes and apply the quorum
def quorum_result(votes, threshold=4):
    approvals = sum(vote is True for vote in votes.values())
    responding = sum(vote is not None for vote in votes.values())
    return approvals >= threshold, approvals, responding

votes = {
    validator: validate_proposal(proposal)[0]
    for validator in validators
}

print(votes)
print(quorum_result(votes))

{'University A': True, 'University B': True, 'University C': True, 'University D': True, 'University E': True}
(True, 5, 5)


> **Expected Result:** All five validators approve, so the proposal reaches the four-of-five quorum (`(True, 5, 5)`).

## Part 2: Crash and Byzantine faults
A crash fault means that a validator stops responding. A Byzantine validator can behave arbitrarily, including rejecting valid data, approving invalid data, or sending contradictory messages.

In [2]:
# Step 1: Simulate one crashed validator
crash_votes = votes.copy()
crash_votes["University E"] = None

print(crash_votes)
print(quorum_result(crash_votes))

{'University A': True, 'University B': True, 'University C': True, 'University D': True, 'University E': None}
(True, 4, 4)


> **Expected Result:** Four validators still approve. The proposal can progress despite one unavailable participant (`(True, 4, 4)`).

In [3]:
# Step 2: Simulate one dishonest validator (maliciously rejects valid proposal)
dishonest_votes = votes.copy()
dishonest_votes["University E"] = False

print(dishonest_votes)
print(quorum_result(dishonest_votes))

{'University A': True, 'University B': True, 'University C': True, 'University D': True, 'University E': False}
(True, 4, 5)


> **Expected Result:** The four honest approvals still satisfy the quorum (`(True, 4, 5)`).

In [4]:
# Step 3: Submit an invalid proposal
invalid_proposal = {
    "student": "Marta Kowalska",
    "credential": "Blockchain Fundamentals",
    "issuer": "Unknown University",
}

invalid_votes = {
    validator: validate_proposal(invalid_proposal)[0]
    for validator in validators
}
# University E maliciously votes True
invalid_votes["University E"] = True

print(invalid_votes)
print(quorum_result(invalid_votes))

{'University A': False, 'University B': False, 'University C': False, 'University D': False, 'University E': True}
(False, 1, 5)


> **Expected Result:** One dishonest approval cannot make the invalid proposal reach a four-of-five quorum (`(False, 1, 5)`).

## Part 3: Network partitions safety and liveness
A network partition separates validators into groups that cannot communicate. A high threshold can prevent conflicting decisions, but it may also stop the system from making progress until communication returns.

In [5]:
# Step 1: Test a three-two partition
def has_quorum(voter_set, threshold):
    return len(set(voter_set)) >= threshold

left_side = {"University A", "University B", "University C"}
right_side = {"University D", "University E"}

print("Left at threshold 4:", has_quorum(left_side, 4))
print("Right at threshold 4:", has_quorum(right_side, 4))

Left at threshold 4: False
Right at threshold 4: False


> **Observation:** Neither side reaches four votes. The ledger pauses, which protects safety but reduces liveness.

In [6]:
# Step 2: Lower the threshold and add equivocation
proposal_x_voters = {
    "University A", "University B", "University E"
}
proposal_y_voters = {
    "University C", "University D", "University E"
}

print("X at threshold 3:", has_quorum(proposal_x_voters, 3))
print("Y at threshold 3:", has_quorum(proposal_y_voters, 3))
print("Voted for both:", proposal_x_voters & proposal_y_voters)

X at threshold 3: True
Y at threshold 3: True
Voted for both: {'University E'}


> **Observation:** University E sends contradictory votes. With a threshold of three, both conflicting proposals appear to reach quorum. This simplified experiment shows why quorum intersection and the assumed number of Byzantine validators matter.

### Step 3: Compare the thresholds

| Threshold | Progress during a three-two partition | Conflicting quorum risk in this experiment |
| :--- | :--- | :--- |
| **4 of 5** | Neither side finalizes (pauses progress) | One equivocator cannot finalize both proposals (preserves **safety**) |
| **3 of 5** | The three-node side can finalize (maintains **liveness**) | One equivocator can support two conflicting groups (violates **safety**) |

#### Short Explanation of Safety vs. Liveness:
- **Safety** guarantees that *nothing bad happens* (e.g., conflicting ledger histories, double spending, or two mutually exclusive proposals finalized).
- **Liveness** guarantees that *something good eventually happens* (e.g., valid proposals continue to be processed and finality is reached without the system stalling).
- With a **4-of-5 threshold**, safety is prioritized over liveness during a partition: neither side finalizes conflicting updates. With a **3-of-5 threshold**, liveness is prioritized, but safety is sacrificed if a Byzantine node equivocates.

## Part 4: Sybil resistance
An open system cannot safely count every key or node as one independent voter. One actor can create many inexpensive identities and obtain an artificial majority.

In [7]:
# Step 1: Count identities without membership control
def simple_majority(votes):
    approvals = sum(votes.values())
    return approvals > len(votes) / 2

honest_votes = {
    validator: False for validator in validators
}
fake_votes = {
    f"Fake identity {number}": True
    for number in range(1, 21)
}

open_votes = {**honest_votes, **fake_votes}

print("Identity count:", len(open_votes))
print("Attacker wins:", simple_majority(open_votes))

Identity count: 25
Attacker wins: True


> **Expected Result:** Twenty fake identities outvote the five consortium members (`Attacker wins: True`).

In [8]:
# Step 2: Apply permissioned membership
authorized = set(validators)
authorized_votes = {
    identity: vote
    for identity, vote in open_votes.items()
    if identity in authorized
}

print("Authorized identities:", len(authorized_votes))
print("Attacker wins:", simple_majority(authorized_votes))

Authorized identities: 5
Attacker wins: False


> **Expected Result:** The fake identities have no voting rights. A permissionless blockchain cannot rely on a membership list, so it must weight a scarce resource such as computation or stake.

### Step 3: Explain the mechanism
- **Permissioned consortium:** Authenticated membership limits who votes using cryptographic authorization (PKI). Fake identities are simply ignored regardless of how many are spawned.
- **Proof of Work (PoW):** Influence follows demonstrated computation (hash power) rather than identity count. Spawning 1,000 virtual identities does not increase total hash power.
- **Proof of Stake (PoS):** Influence follows economic stake controlled under protocol rules. Voting weight is tied to capital deposited, rendering identity multiplicity ineffective.

## Part 5: Effective control and unequal voting power
A system can display many validators while a small coalition controls enough voting weight to decide outcomes. Measure the minimum coalition that can reach a chosen threshold.

In [9]:
# Step 1: Add the coalition function
def minimum_coalition(weights, threshold):
    ordered = sorted(
        weights.items(), key=lambda item: item[1], reverse=True
    )
    coalition = []
    total = 0
    for validator, weight in ordered:
        coalition.append(validator)
        total += weight
        if total >= threshold:
            return coalition, total
    return coalition, total

# Step 2: Compare equal and concentrated power (Threshold = 67)
equal_weights = {
    "A": 20, "B": 20, "C": 20, "D": 20, "E": 20
}
concentrated_weights = {
    "A": 40, "B": 25, "C": 15, "D": 10, "E": 10
}

print("Equal:", minimum_coalition(equal_weights, 67))
print("Concentrated:", minimum_coalition(concentrated_weights, 67))

# Step 3: Create an extreme case
# Validator A has 70 weight, remaining 30 divided among others (10, 10, 5, 5)
extreme_weights = {
    "A": 70, "B": 10, "C": 10, "D": 5, "E": 5
}
print("Extreme:", minimum_coalition(extreme_weights, 67))

Equal: (['A', 'B', 'C', 'D'], 80)
Concentrated: (['A', 'B', 'C'], 80)
Extreme: (['A'], 70)


> **Expected Result & Analysis for Step 3:**
- **Equal Weights:** 4 validators are needed (`['A', 'B', 'C', 'D']`, total weight 80) to reach the 67 threshold.
- **Concentrated Weights:** 3 validators suffice (`['A', 'B', 'C']`, total weight 80).
- **Extreme Weights:** Only 1 validator is needed (`['A']`, total weight 70).

**Does validator count still describe decentralization accurately?**  
**No.** Validator count alone is misleading. In the extreme case, the system displays 5 validators on paper, suggesting a decentralized network; however, Validator A holds 70% of the voting power and can unilaterally approve or block any proposal without consulting any other participant. Effective control is completely centralized in a single entity (a dictatorship). True decentralization must be evaluated by the distribution of voting weight, the minimum coalition required to dictate decisions (Nakamoto coefficient), and resistance to colluding subsets.

## Part 6: Choosing a decentralization design

### Summary of Scenarios

| Scenario | Starting point | Question to resolve | Key Architectural Choice |
| :--- | :--- | :--- | :--- |
| **Five universities share micro-credentials** | Permissioned consortium | Which members validate and what quorum applies | All 5 universities validate; 4-of-5 quorum prevents conflicting updates and tolerates 1 Byzantine fault. |
| **An open digital asset must resist issuer censorship** | Permissionless blockchain | Which scarce resource provides Sybil resistance | Proof of Work (hashpower) or Proof of Stake (bonded capital) to allow anonymous, permissionless participation. |
| **One hospital records internal equipment maintenance** | Conventional signed database | What benefit would a blockchain add | A centralized database with cryptographic audit logs (append-only) suffices; a distributed blockchain introduces unnecessary overhead since there is only one administrative domain. |

---

### Final Task: Short Design Proposal
**Selected Scenario:** *Five universities share micro-credentials*

#### 1. Who may submit records and who validates them
- **Submission:** Registered registrar offices and authorized academic departments of the five member universities may submit credential issuance records using their institution-issued digital keys.
- **Validation:** All five designated university nodes (University A through E) operate full validator nodes. Each validator verifies cryptographic signatures, issuer authenticity against the consortium whitelist, and schema compliance.

#### 2. Who may read the ledger and who governs membership or upgrades
- **Read Access:** Publicly readable. Students, graduates, employers, and educational institutions worldwide can freely query the ledger and verify credential proofs without permission.
- **Governance:** Governed by an on-chain/off-chain consortium council consisting of one designated representative per university. Adding/removing universities or upgrading smart contract schemas requires a supermajority vote (at least 4 out of 5 universities).

#### 3. Which failures or collusion the design must tolerate
- The design must tolerate **1 crash fault** (offline node during maintenance/downtime) and **1 Byzantine fault** (a compromised or malicious university attempting to forge credentials or equivocate votes) out of 5 nodes ($N=5, f=1$).

#### 4. How the design resists fake identities
- The consortium operates as a **permissioned network** with strict Public Key Infrastructure (PKI). Only validator nodes with public keys explicitly registered in the genesis validator set or admitted via governance votes have voting power. Any unauthorized fake identity is immediately discarded during membership filtering (as demonstrated in Part 4).

#### 5. What quorum, fork-choice, or finality rule determines acceptance
- A **deterministic BFT quorum threshold of 4-of-5 (80%)** determines acceptance.
- Because two quorums of 4 out of 5 must intersect in at least 3 nodes ($4 + 4 - 5 = 3$), a single Byzantine equivocator cannot create two conflicting finalizations. This ensures **deterministic immediate finality** without chain reorganizations or probabilistic fork choices.

#### 6. Where effective control could become concentrated
- **Unequal Voting Weights:** If voting weight is tied to university size, research funding, or student population rather than equal 1-node-1-vote, a small coalition (or single university) could gain unilateral control (as demonstrated in Part 5).
- **Infrastructure / Cloud Provider Collusion:** If multiple universities host their validator instances on the same cloud provider (e.g., AWS us-east-1) or contract the same IT vendor, an outage or provider-level compromise could halt or manipulate the consensus.
- **Private Key Centralization:** Inadequate key management policies could lead to multi-node credential leaks.

#### 7. Whether a simpler signed database would meet the same need
- While a conventional centralized signed database (e.g., PostgreSQL with digital signatures) is simpler and faster, it requires selecting **one single organization** to host and manage the database. That host university would possess unilateral power to censor rival institutions, delete entries, or alter operational rules, creating an unacceptable conflict of interest.
- A **permissioned consortium blockchain** distributes trust symmetrically across all five institutions, ensuring multi-party validation, tamper-evident auditability, and no single point of failure or administrative censorship.